In [46]:
#importing stuff
import numpy as np                                      
import pandas as pd
import matplotlib.pyplot as plt
import math,os,random,sys,h5py
from sklearn import preprocessing
from sklearn.svm import SVR
from sklearn.impute import SimpleImputer
from rdkit import Chem
from mordred import Calculator, descriptors
from sklearn.exceptions import ConvergenceWarning

In [47]:
APs = pd.read_csv("tripeptide_AP.txt", sep=": ", index_col=0, header=None, engine = 'python')       #AP scores from simulation
f = h5py.File('tripeptides.hdf5','r')                                                               #reading data from the file generated from judred generator
peps = np.array(f.get('peptides'))
feas = np.array(f.get('features'))
data = np.array(f.get('data'))
f.close()
judp = pd.DataFrame(data, columns = feas, index = peps)                                             #arranging it into a pandas data frame


In [48]:
x_train = [np.array(judp.loc[b'ALA-ALA-ALA'])]                  # initializing the training data with data of polyalanine 
Y_train = np.array(APs.loc['ALA-ALA-ALA'])
print(x_train)
print(Y_train)

[array([  0.     ,   0.     , 231.26942,   0.     ,  -0.99   ,   0.     ,
       387.     ,   0.     ,  34.5    ,   0.     ], dtype=float32)]
[1.64890034]


In [49]:
def randPepSelector(APs, judp, x_train, Y_train):                           # a is APs, b is judp/modp
    for pep in range(10):                                              # generates random indecies, looks up those indecies data and adds to the training data
        i = random.randrange(1,8000)
        p = APs.iloc[i]
        Y_train = np.append(Y_train, np.array(p), axis=0)
        x_train = np.append(x_train, [np.array(judp.loc[p.name.encode('utf-8')])],axis=0)
    return Y_train,x_train       

In [50]:
Y_train,x_train = randPepSelector(APs,judp,x_train,Y_train)                 # first we take a set of random peptides(10)
print(x_train)
print(Y_train)

[[ 0.0000000e+00  0.0000000e+00  2.3126942e+02  0.0000000e+00
  -9.9000001e-01  0.0000000e+00  3.8700000e+02  0.0000000e+00
   3.4500000e+01  0.0000000e+00]
 [ 7.0000000e+00  0.0000000e+00  3.7739944e+02  0.0000000e+00
  -1.5199999e+00 -1.0000000e+00  5.9200000e+02  1.4000000e+00
   4.8910000e+01  0.0000000e+00]
 [ 3.0000000e+00  0.0000000e+00  3.5743945e+02  1.0000000e+00
   3.6999997e-01  0.0000000e+00  5.6500000e+02  6.0000002e-01
   4.8719997e+01  0.0000000e+00]
 [ 1.0000000e+00  1.0000000e+00  3.1832944e+02  0.0000000e+00
  -2.2999999e-01  0.0000000e+00  5.2400000e+02  2.0000000e-01
   4.3860001e+01  1.0000000e+00]
 [ 2.0000000e+00  0.0000000e+00  3.1927945e+02  0.0000000e+00
  -5.1600003e+00 -2.0000000e+00  5.2000000e+02  6.6666669e-01
   2.8650000e+01  0.0000000e+00]
 [ 9.0000000e+00  1.0000000e+00  4.4552945e+02  0.0000000e+00
   8.6000001e-01  0.0000000e+00  7.0700000e+02  1.2857143e+00
   5.7519997e+01  0.0000000e+00]
 [ 2.0000000e+00  4.0000000e+00  4.3352948e+02  1.0000000e

In [51]:
def loc(s,Y_pred):                      #takes the APs of top peptides and returns their location in judred parameters so we can look up their names by their location
    y = []
    for i in range(len(s)):
        for j in range(len(Y_pred)):
            if s[i] == Y_pred[j]:
                if j not in y:
                    y.append(j)
    return y


In [52]:
def topN(n,judp,x_train,Y_train):                                                                #takes the judred parameters and the training data 
    svr = SVR(kernel='rbf',gamma='scale',C=100,epsilon=0.1,max_iter=-1,tol=0.0001,verbose=0)    #trains a svm(rbf)
    svr.fit(x_train,Y_train)
    x = []
    for i in range(8000):
        x.append(np.array(judp.iloc[i]))
    imputer = SimpleImputer(strategy='mean')                                                    #imputer removes NaN values from the data set
    x_imputed = imputer.fit_transform(x)
    Y_pred = svr.predict(x_imputed)                                                             #predicts AP scores for all peptides
    s = sorted(Y_pred)
    s = s[8000-n:]
    y_loc = loc(s,Y_pred)
    y_nam = ['']*len(y_loc)
    for i in range(len(y_loc)):
        y_nam[i] = judp.iloc[y_loc[i]].name
    return y_nam,y_loc                                                                          #returns the top N peptides by their name and location in judp


In [53]:
def addToTrain(y_nam,y_loc,APs,judp,x_train,Y_train):                                               #takes the predicted peptide in previous iteration and adds them to training data
    for i in range(len(y_loc)):
        x_train = np.append(x_train, [np.array(judp.iloc[y_loc[i]])], axis=0)
    for i in y_nam:
        Y_train = np.append(Y_train, np.array(APs.loc[i.decode(encoding='utf-8')]), axis=0)
    return x_train,Y_train

In [54]:
def runAL(num_iter,APs,judp,x_train,Y_train):
    for i in range(num_iter):
        y_nam,y_loc = topN(10,judp,x_train,Y_train)
        x_train,Y_train = addToTrain(y_nam,y_loc,APs,judp,x_train,Y_train)
    return topN(40,judp,x_train,Y_train)

y_nam,y_loc = runAL(80,APs,judp,x_train,Y_train)
print(y_nam)
        

[b'TRP-PHE-PRO', b'PHE-PRO-TRP', b'PHE-TRP-PRO', b'PRO-PHE-TRP', b'PRO-TRP-ILE', b'TRP-PRO-ILE', b'ILE-PRO-TRP', b'ILE-TRP-PRO', b'TRP-ILE-PRO', b'PRO-ILE-TRP', b'PRO-PRO-TRP', b'PRO-TRP-PRO', b'TRP-PRO-PRO', b'TRP-PHE-VAL', b'PHE-VAL-TRP', b'PHE-TRP-VAL', b'VAL-PHE-TRP', b'VAL-TRP-PHE', b'TRP-VAL-PHE', b'ILE-TRP-VAL', b'VAL-ILE-TRP', b'VAL-TRP-ILE', b'TRP-ILE-VAL', b'TRP-VAL-ILE', b'ILE-VAL-TRP', b'PRO-TRP-VAL', b'TRP-PRO-VAL', b'PRO-VAL-TRP', b'VAL-PRO-TRP', b'VAL-TRP-PRO', b'TRP-VAL-PRO', b'PRO-TRP-TRP', b'TRP-PRO-TRP', b'TRP-TRP-PRO', b'VAL-TRP-TRP', b'TRP-VAL-TRP', b'TRP-TRP-VAL', b'VAL-VAL-TRP', b'VAL-TRP-VAL', b'TRP-VAL-VAL']
